In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [ ]:
import pandas as pd
import numpy as np
import re

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

uploaded = files.upload()

icd10_file = None

for name in uploaded.keys():
    lower = name.lower()
    if "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

print("Detected ICD-10 file:", icd10_file)

if icd10_file is None:
    raise ValueError("Please upload processed_icd10_reference_valid_only.csv")

icd10_df = pd.read_csv(icd10_file)

print("icd10_df shape:", icd10_df.shape)
print("Columns:", icd10_df.columns.tolist())
display(icd10_df.head())

Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1).csv
Detected ICD-10 file: processed_icd10_reference_valid_only (1).csv
icd10_df shape: (9943, 10)
Columns: ['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


In [ ]:
# Clean rows
icd10_df = icd10_df.dropna(subset=["Literal_match", "y_category"]).copy()
icd10_df = icd10_df[icd10_df["Literal_match"].astype(str).str.len() > 0].copy()
icd10_df = icd10_df[icd10_df["y_category"].astype(str).str.len() > 0].copy()

# Same split logic as before
category_counts = icd10_df["y_category"].value_counts()
rare_categories = category_counts[category_counts < 2].index

rare_df = icd10_df[icd10_df["y_category"].isin(rare_categories)].copy()
normal_df = icd10_df[~icd10_df["y_category"].isin(rare_categories)].copy()

train_normal, val_part = train_test_split(
    normal_df,
    test_size=0.2,
    random_state=42,
    stratify=normal_df["y_category"]
)

train_part = pd.concat([train_normal, rare_df], ignore_index=True)

print("train_part:", train_part.shape)
print("val_part:", val_part.shape)

train_part: (7954, 10)
val_part: (1989, 10)


In [ ]:
X_train = train_part["Literal_match"].fillna("").astype(str)
y_train = train_part["y_category"].astype(str)

X_val = val_part["Literal_match"].fillna("").astype(str)
y_val = val_part["y_category"].astype(str)

baseline_svc = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
            lowercase=True
        ))
    ])),
    ("clf", LinearSVC(
        C=0.5,
        class_weight=None,
        max_iter=20000,
        random_state=42
    ))
])

baseline_svc.fit(X_train, y_train)

val_part["base_svc_pred"] = baseline_svc.predict(X_val)

print("Validation accuracy:", accuracy_score(y_val, val_part["base_svc_pred"]))
display(val_part[["Literal_basic", "Literal_match", "y_category", "base_svc_pred"]].head())

Validation accuracy: 0.7596782302664655


,Literal_basic,Literal_match,y_category,base_svc_pred
8643,anti-d,anti-d,3,3
3555,: Vejiga hiperactiva,vejiga hiperactiva,N,N
3644,RNM cerebral,rnm cerebral,B,B
2726,GSRh: A positiu,gsrh a positiu,Z,Z
910,esplenectomía,esplenectomia,Z,Z


In [ ]:
val_part.to_csv("validation_predictions_main_models.csv", index=False)
files.download("validation_predictions_main_models.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# Upload:
# 1. validation_predictions_main_models.csv
# 2. processed_icd10_reference_valid_only.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

val_file = None
icd10_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "validation_predictions_main_models" in lower or "validation_predictions" in lower:
        val_file = name

    elif "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

print("Detected validation predictions file:", val_file)
print("Detected ICD-10 processed file:", icd10_file)

if val_file is None:
    raise ValueError("Could not detect validation_predictions_main_models.csv. Please upload it.")

if icd10_file is None:
    raise ValueError("Could not detect processed_icd10_reference_valid_only.csv. Please upload it.")

val_part = pd.read_csv(val_file)
icd10_df = pd.read_csv(icd10_file)

print("val_part shape:", val_part.shape)
print("icd10_df shape:", icd10_df.shape)

print("val_part columns:")
print(val_part.columns.tolist())

print("icd10_df columns:")
print(icd10_df.columns.tolist())

display(val_part.head())
display(icd10_df.head())

Saving validation_predictions_main_models.csv to validation_predictions_main_models (1).csv
Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1) (1).csv
Uploaded files:
validation_predictions_main_models (1).csv
processed_icd10_reference_valid_only (1) (1).csv
Detected validation predictions file: validation_predictions_main_models (1).csv
Detected ICD-10 processed file: processed_icd10_reference_valid_only (1) (1).csv
val_part shape: (1989, 11)
icd10_df shape: (9943, 10)
val_part columns:
['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match', 'base_svc_pred']
icd10_df columns:
['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match,base_svc_pred
0,3E0234Z,anti-d,3E0234Z,anti-d,True,P,"Introducción en músculo de suero, toxoide y va...",3,3E0234Z,anti-d,3
1,N3281,: Vejiga hiperactiva,N3281,: Vejiga hiperactiva,True,D,Vejiga hiperactiva,N,N3281,vejiga hiperactiva,N
2,B030ZZZ,RNM cerebral,B030ZZZ,RNM cerebral,True,P,Imagen por resonancia magnética (rm) de cerebro,B,B030ZZZ,rnm cerebral,B
3,Z6710,GSRh: A positiu,Z6710,GSRh: A positiu,True,D,"Grupo sanguíneo A, Rh positivo",Z,Z6710,gsrh a positiu,Z
4,Z9081,esplenectomía,Z9081,esplenectomía,True,D,Ausencia adquirida de bazo,Z,Z9081,esplenectomia,Z


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


In [ ]:
error_df = val_part.copy()

error_df["true_category"] = error_df["y_category"].astype(str)
error_df["pred_category"] = error_df["base_svc_pred"].astype(str)
error_df["is_correct"] = error_df["true_category"] == error_df["pred_category"]

print("Accuracy:", error_df["is_correct"].mean())
print("Wrong rows:", (~error_df["is_correct"]).sum())

Accuracy: 0.7596782302664655
Wrong rows: 478


In [ ]:
error_df = val_part.copy()

error_df["true_category"] = error_df["y_category"].astype(str)
error_df["pred_category"] = error_df["base_svc_pred"].astype(str)
error_df["is_correct"] = error_df["true_category"] == error_df["pred_category"]

print("Accuracy:", error_df["is_correct"].mean())
print("Wrong rows:", (~error_df["is_correct"]).sum())

Accuracy: 0.7596782302664655
Wrong rows: 478


In [ ]:
literal_train_counts = train_part["Literal_match"].value_counts()

literal_cat_counts = (
    train_part
    .groupby(["Literal_match", "y_category"])
    .size()
    .reset_index(name="count")
)

literal_totals = (
    literal_cat_counts
    .groupby("Literal_match")["count"]
    .sum()
    .reset_index(name="total_count")
)

literal_cat_counts = literal_cat_counts.merge(literal_totals, on="Literal_match", how="left")
literal_cat_counts["purity"] = literal_cat_counts["count"] / literal_cat_counts["total_count"]

literal_majority = (
    literal_cat_counts
    .sort_values(["Literal_match", "count"], ascending=[True, False])
    .drop_duplicates("Literal_match")
)

error_df["literal_train_count"] = error_df["Literal_match"].map(literal_train_counts).fillna(0).astype(int)
error_df["literal_train_purity"] = error_df["Literal_match"].map(
    literal_majority.set_index("Literal_match")["purity"]
).fillna(0)

error_df["literal_majority_category"] = error_df["Literal_match"].map(
    literal_majority.set_index("Literal_match")["y_category"]
)

error_df["is_ambiguous_literal"] = (
    (error_df["literal_train_count"] > 1) &
    (error_df["literal_train_purity"] < 1.0)
)

In [ ]:
def text_shape_features(text):
    text = str(text)
    return pd.Series({
        "char_len": len(text),
        "word_count": len(text.split()),
        "has_digit": bool(re.search(r"\d", text)),
        "has_plus": "+" in text,
        "has_dot": "." in text,
        "has_dash": "-" in text,
        "has_slash": "/" in text,
        "is_short_4": len(text) <= 4,
        "looks_code_like": bool(re.search(r"\b[a-z]?\d+(\.\d+)?[a-z]*\b", text))
    })

shape = error_df["Literal_match"].apply(text_shape_features)
error_df = pd.concat([error_df, shape], axis=1)

In [ ]:
wrong_df = error_df[~error_df["is_correct"]].copy()

confusion_pairs = (
    wrong_df
    .groupby(["true_category", "pred_category"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(confusion_pairs.head(30))

,true_category,pred_category,count
190,Z,O,17
59,E,O,15
54,D,O,14
126,O,E,14
66,F,O,14
74,G,O,10
174,V,Z,10
180,Z,0,10
12,0,Z,9
194,Z,V,9


In [ ]:
import os
import pandas as pd
import numpy as np
import re

# Make outputs folder if it does not exist
os.makedirs("outputs", exist_ok=True)

# ------------------------------------------------------------
# Create error_df
# ------------------------------------------------------------
error_df = val_part.copy()

error_df["true_category"] = error_df["y_category"].astype(str)
error_df["pred_category"] = error_df["base_svc_pred"].astype(str)
error_df["is_correct"] = error_df["true_category"] == error_df["pred_category"]

print("Validation accuracy:", error_df["is_correct"].mean())
print("Total rows:", len(error_df))
print("Wrong rows:", (~error_df["is_correct"]).sum())

# ------------------------------------------------------------
# Literal frequency and ambiguity
# ------------------------------------------------------------
literal_train_counts = train_part["Literal_match"].value_counts()
category_train_counts = train_part["y_category"].astype(str).value_counts()

error_df["literal_train_count"] = (
    error_df["Literal_match"]
    .map(literal_train_counts)
    .fillna(0)
    .astype(int)
)

error_df["true_category_train_count"] = (
    error_df["true_category"]
    .map(category_train_counts)
    .fillna(0)
    .astype(int)
)

literal_cat_counts = (
    train_part
    .groupby(["Literal_match", "y_category"])
    .size()
    .reset_index(name="count")
)

literal_total = (
    literal_cat_counts
    .groupby("Literal_match")["count"]
    .sum()
    .reset_index(name="total_count")
)

literal_cat_counts = literal_cat_counts.merge(
    literal_total,
    on="Literal_match",
    how="left"
)

literal_cat_counts["purity"] = (
    literal_cat_counts["count"] / literal_cat_counts["total_count"]
)

literal_majority = (
    literal_cat_counts
    .sort_values(["Literal_match", "count"], ascending=[True, False])
    .drop_duplicates("Literal_match")
)

literal_purity_map = literal_majority.set_index("Literal_match")["purity"]
literal_majority_cat_map = literal_majority.set_index("Literal_match")["y_category"]

error_df["literal_train_purity"] = (
    error_df["Literal_match"]
    .map(literal_purity_map)
    .fillna(0)
)

error_df["literal_majority_category"] = (
    error_df["Literal_match"]
    .map(literal_majority_cat_map)
)

error_df["is_ambiguous_literal"] = (
    (error_df["literal_train_count"] > 1) &
    (error_df["literal_train_purity"] < 1.0)
)

# ------------------------------------------------------------
# Text shape features
# ------------------------------------------------------------
def text_shape_features(text):
    text = str(text)
    tokens = text.split()
    return pd.Series({
        "char_len": len(text),
        "word_count": len(tokens),
        "has_digit": bool(re.search(r"\d", text)),
        "has_plus": "+" in text,
        "has_dot": "." in text,
        "has_dash": "-" in text,
        "has_slash": "/" in text,
        "is_short_4": len(text) <= 4,
        "looks_code_like": bool(re.search(r"\b[a-z]?\d+(\.\d+)?[a-z]*\b", text))
    })

shape_features = error_df["Literal_match"].apply(text_shape_features)

Validation accuracy: 0.7596782302664655
Total rows: 1989
Wrong rows: 478


In [ ]:
def text_shape_features(text):
    text = str(text)
    tokens = text.split()

    return pd.Series({
        "char_len": len(text),
        "word_count": len(tokens),
        "has_digit": bool(re.search(r"\d", text)),
        "has_plus": "+" in text,
        "has_dot": "." in text,
        "has_dash": "-" in text,
        "has_slash": "/" in text,
        "is_short_4": len(text) <= 4,
        "looks_code_like": bool(re.search(r"\b[a-z]?\d+(\.\d+)?[a-z]*\b", text))
    })

shape_features = error_df["Literal_match"].apply(text_shape_features)

# Avoid duplicate columns if rerunning the cell
shape_cols = shape_features.columns.tolist()
error_df = error_df.drop(columns=[c for c in shape_cols if c in error_df.columns], errors="ignore")

error_df = pd.concat([error_df, shape_features], axis=1)

print("Shape features added.")
display(error_df[[
    "Literal_basic",
    "Literal_match",
    "true_category",
    "pred_category",
    "is_correct",
    "char_len",
    "word_count",
    "has_digit",
    "is_short_4",
    "looks_code_like"
]].head())

Shape features added.


,Literal_basic,Literal_match,true_category,pred_category,is_correct,char_len,word_count,has_digit,is_short_4,looks_code_like
0,anti-d,anti-d,3,3,True,6,1,False,False,False
1,: Vejiga hiperactiva,vejiga hiperactiva,N,N,True,18,2,False,False,False
2,RNM cerebral,rnm cerebral,B,B,True,12,2,False,False,False
3,GSRh: A positiu,gsrh a positiu,Z,Z,True,14,3,False,False,False
4,esplenectomía,esplenectomia,Z,Z,True,13,1,False,False,False


In [ ]:
# ------------------------------------------------------------
# Assign error cause
# ------------------------------------------------------------
def assign_error_cause(row):
    if row["is_correct"]:
        return "correct"

    if row["is_ambiguous_literal"]:
        return "ambiguous_literal_same_text_multiple_labels"

    if row["literal_train_count"] == 0:
        return "unseen_literal"

    if row["is_short_4"]:
        return "short_or_abbreviation"

    if row["looks_code_like"]:
        return "code_like_or_numeric_literal"

    if row["pred_category"] in ["O", "Z"]:
        return f"overpredicted_{row['pred_category']}"

    return "other_error"

error_df["error_cause"] = error_df.apply(assign_error_cause, axis=1)

error_cause_summary = (
    error_df
    .groupby("error_cause")
    .agg(
        rows=("Literal_match", "count"),
        accuracy=("is_correct", "mean"),
        wrong=("is_correct", lambda x: (~x).sum())
    )
    .reset_index()
    .sort_values("wrong", ascending=False)
)

display(error_cause_summary)

# ------------------------------------------------------------
# Confusion pairs
# ------------------------------------------------------------
wrong_df = error_df[~error_df["is_correct"]].copy()

confusion_pairs = (
    wrong_df
    .groupby(["true_category", "pred_category"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(confusion_pairs.head(30))

,error_cause,rows,accuracy,wrong
7,unseen_literal,260,0.0,260
0,ambiguous_literal_same_text_multiple_labels,136,0.0,136
3,other_error,41,0.0,41
4,overpredicted_O,19,0.0,19
5,overpredicted_Z,11,0.0,11
1,code_like_or_numeric_literal,9,0.0,9
6,short_or_abbreviation,2,0.0,2
2,correct,1511,1.0,0


,true_category,pred_category,count
190,Z,O,17
59,E,O,15
54,D,O,14
126,O,E,14
66,F,O,14
74,G,O,10
174,V,Z,10
180,Z,0,10
12,0,Z,9
194,Z,V,9


In [ ]:
import os
from google.colab import files

os.makedirs("outputs", exist_ok=True)

error_cause_summary.to_csv("outputs/error_cause_summary.csv", index=False)
confusion_pairs.to_csv("outputs/confusion_pairs.csv", index=False)
error_df.to_csv("outputs/error_analysis_full.csv", index=False)

files.download("outputs/error_cause_summary.csv")
files.download("outputs/confusion_pairs.csv")
files.download("outputs/error_analysis_full.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>